# Нейросетевые модели

## Часть 2.4: Обнаружение объектов на изображении - обучение YOLO

### 1. Данные для обучения

Будем работать с данными для обнаружения дефектов на сварных швах. Задача состоит в том, что необходимо:
1. обнаружить изделие на снимке
2. обнаружить сварной шов на изделии
3. обнаружить (если есть) дефекты на сварном шве

Данные есть в виде набора снимков металлических изделий с уже размеченными изделиями, сварными швами и дефектами.

In [ ]:
!ls -la /home/jovyan/__DATA/GZP/weldingdata

In [ ]:
!ls -la /home/jovyan/__DATA/GZP/weldingdata/test | head -n 8

In [ ]:
!cat /home/jovyan/__DATA/GZP/weldingdata/test/_annotations.csv | head -n 5

### 2. Постановка задачи

Для обучения модели воспользуемся помощью ИИ, поставим задачу с учетом ряда требований к реализации:

```prompt
## Базовые параметры
- роль: инженер по компьютерному зрению на Python
- уровень: продвинутый
- температура: 0 (максимальная точность и предсказуемость)

## Входные данные
- набор данных (изображения, разметка) по сварным швам на металлических изделиях

## Задача
- обучить модель YOLO находить швы и дефекты на изображениях, то есть решить задачу jbject detection применительно к имеющемуся набору данных

## Требования к реализации
- предобработать данные и привести их в нужный формат и структуру для дообучения модели
- использовать обучающую, валидационную и тестовую выборки
- визуализировать несколько образцов обучающих данных, если классов несколько, то вывести образцы со всеми классами
- сформировать необходимые артефакты для обучения в виде YAML файлов
- протестировать обученую модель на тестовой выборке с выборочной визуализацией результатов

### Технические ограничения
- YOLO модель `yolo26n.pt` от Ultralytics
- COCO формат для работы с данными
- библиотеки opencv и PIL для обработки изображений
- оптимизация алгоритма под CPU для максимальной скорости обучения

## Технические требования
- архитектура: максимально упрощенная
- вывод данных: адаптированный под интерактивные ноутбуки, где возможно используй display вместо print 
- стиль кода: PEP 8, black (длина строки 79)
- документация: Docstrings в стиле Google на русском языке

### Файловая структура входных данных
$ls -la /home/jovyan/__DATA/GZP/weldingdata
total 0
drwxrwxrwx 2 root root 0 Apr 20 14:45 .
drwxrwxrwx 2 root root 0 Apr 22 07:15 ..
drwxrwxrwx 2 root root 0 Apr 20 13:12 test
drwxrwxrwx 2 root root 0 Apr 20 14:45 train
drwxrwxrwx 2 root root 0 Apr 20 13:12 valid

$ls -la /home/jovyan/__DATA/GZP/weldingdata/train | head -n 8
total 39104
drwxrwxrwx 2 root root      0 Apr 20 14:45 .
drwxrwxrwx 2 root root      0 Apr 20 14:45 ..
-rw-rw-rw- 1 root root  46474 Apr 20 13:12 SampleV1_1_mp4-0_jpg.rf.65486f04c6e53e160a2abf2bb33d012f.jpg
-rw-rw-rw- 1 root root  44927 Apr 20 13:12 SampleV1_1_mp4-0_jpg.rf.f12530bc5f266d5e859cd29d6342a673.jpg
-rw-rw-rw- 1 root root  42741 Apr 20 13:12 SampleV1_1_mp4-0_jpg.rf.f927c0ce54516109c97fff58f5d39038.jpg
-rw-rw-rw- 1 root root  44870 Apr 20 13:12 SampleV1_1_mp4-10_jpg.rf.63a9070cfa108fe21921e4bc94da6d06.jpg

$ls -la /home/jovyan/__DATA/GZP/weldingdata/test | head -n 8
total 1022
drwxrwxrwx 2 root root     0 Apr 20 13:12 .
drwxrwxrwx 2 root root     0 Apr 20 14:45 ..
-rw-rw-rw- 1 root root 43418 Apr 20 13:12 SampleV1_1_mp4-1_jpg.rf.3f50c974a91c4e6348dd49491f06def8.jpg
-rw-rw-rw- 1 root root 43397 Apr 20 13:12 SampleV1_1_mp4-42_jpg.rf.6e68d9186e630ffb996233ad2a593f51.jpg
-rw-rw-rw- 1 root root 39437 Apr 20 13:12 SampleV1_2_mp4-14_jpg.rf.4dba7e8bd84314a155dd85df33b5f4d9.jpg
-rw-rw-rw- 1 root root 50271 Apr 20 13:12 SampleV2_1_mp4-0_jpg.rf.32dc225d9b5d11d01260f75c918a9961.jpg

$ls -la /home/jovyan/__DATA/GZP/weldingdata/valid | head -n 8
total 3812
drwxrwxrwx 2 root root     0 Apr 20 13:12 .
drwxrwxrwx 2 root root     0 Apr 20 14:45 ..
-rw-rw-rw- 1 root root 41601 Apr 20 13:12 SampleV1_1_mp4-14_jpg.rf.a63c344a8a0ff6183f314d4a5b231a71.jpg
-rw-rw-rw- 1 root root 42056 Apr 20 13:12 SampleV1_1_mp4-17_jpg.rf.a9d2faf4d350854fbc0d2fc2ec7a63f4.jpg
-rw-rw-rw- 1 root root 39377 Apr 20 13:12 SampleV1_1_mp4-18_jpg.rf.cddd27bcf2b930148b836f4554980319.jpg
-rw-rw-rw- 1 root root 39019 Apr 20 13:12 SampleV1_1_mp4-19_jpg.rf.094d362d62a7104ad597f1c8d56b7784.jpg

### Файлы с аннотациями
$cat /home/jovyan/__DATA/GZP/weldingdata/train/_annotations.csv | head -n 5
filename,width,height,class,xmin,ymin,xmax,ymax
SampleV3_2_mp4-23_jpg.rf.d1cfdcce29497d5b67d2763cfd31506f.jpg,640,640,Workpiece,0,188,540,317
SampleV3_2_mp4-23_jpg.rf.d1cfdcce29497d5b67d2763cfd31506f.jpg,640,640,Welding Line,0,236,526,266
SampleV3_2_mp4-23_jpg.rf.d1cfdcce29497d5b67d2763cfd31506f.jpg,640,640,Workpiece,0,438,534,582
SampleV3_2_mp4-23_jpg.rf.d1cfdcce29497d5b67d2763cfd31506f.jpg,640,640,Welding Line,0,499,512,538

$cat /home/jovyan/__DATA/GZP/weldingdata/test/_annotations.csv | head -n 5
filename,width,height,class,xmin,ymin,xmax,ymax
SampleV3_1_mp4-8_jpg.rf.15adb36ac87bf5e5533ced60df5f5635.jpg,640,640,Workpiece,181,22,452,381
SampleV3_1_mp4-8_jpg.rf.15adb36ac87bf5e5533ced60df5f5635.jpg,640,640,Welding Line,294,34,344,369
SampleV3_1_mp4-8_jpg.rf.15adb36ac87bf5e5533ced60df5f5635.jpg,640,640,Workpiece,192,474,461,640
SampleV3_1_mp4-8_jpg.rf.15adb36ac87bf5e5533ced60df5f5635.jpg,640,640,Welding Line,289,483,364,640

$cat /home/jovyan/__DATA/GZP/weldingdata/valid/_annotations.csv | head -n 5
filename,width,height,class,xmin,ymin,xmax,ymax
SampleV1_1_mp4-18_jpg.rf.cddd27bcf2b930148b836f4554980319.jpg,640,640,Workpiece,217,0,550,344
SampleV1_1_mp4-18_jpg.rf.cddd27bcf2b930148b836f4554980319.jpg,640,640,Workpiece,220,449,529,640
SampleV1_1_mp4-18_jpg.rf.cddd27bcf2b930148b836f4554980319.jpg,640,640,Welding Line,325,0,394,327
SampleV1_1_mp4-18_jpg.rf.cddd27bcf2b930148b836f4554980319.jpg,640,640,Welding Line,316,458,377,640
```

### 3. Реализация обучения модели на Python

In [ ]:
!pip install ultralytics opencv-python

In [ ]:
# Блок 1: Импорт необходимых библиотек
import os
import shutil
import random
from pathlib import Path
from typing import List, Tuple, Dict
import pandas as pd
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import cv2
from IPython.display import display, Image as IPImage
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from sklearn.model_selection import train_test_split
import yaml
from ultralytics import YOLO

# Настройка отображения
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)

In [ ]:
# Блок 2: Определение констант и путей
# Конфигурация путей
DATA_ROOT = Path("/home/jovyan/__DATA/GZP/weldingdata")
TRAIN_DIR = DATA_ROOT / "train"
TEST_DIR = DATA_ROOT / "test"
VALID_DIR = DATA_ROOT / "valid"

# Файлы с аннотациями
TRAIN_ANNOTATIONS = TRAIN_DIR / "_annotations.csv"
TEST_ANNOTATIONS = TEST_DIR / "_annotations.csv"
VALID_ANNOTATIONS = VALID_DIR / "_annotations.csv"

# Пути для преобразованных данных в формате YOLO
YOLO_DATA_ROOT = Path("/home/jovyan/yolo_welding_data")
YOLO_TRAIN_DIR = YOLO_DATA_ROOT / "train"
YOLO_VALID_DIR = YOLO_DATA_ROOT / "valid"
YOLO_TEST_DIR = YOLO_DATA_ROOT / "test"

# Словарь классов (будет определен автоматически)
CLASSES = None
CLASS_NAME_TO_ID = {}
CLASS_ID_TO_NAME = {}

# Параметры обучения
MODEL_NAME = "yolo11n.pt"  # Используем YOLO11n (преемник yolo26n)
EPOCHS = 4
BATCH_SIZE = 8
IMAGE_SIZE = 640
DEVICE = "cpu"  # Оптимизация под CPU

In [ ]:
# Блок 3: Анализ структуры данных и классов
def analyze_dataset_structure():
    """Анализирует структуру датасета и определяет классы"""
    global CLASSES, CLASS_NAME_TO_ID, CLASS_ID_TO_NAME
    
    print("=" * 60)
    print("АНАЛИЗ СТРУКТУРЫ ДАТАСЕТА")
    print("=" * 60)
    
    # Чтение аннотаций
    train_df = pd.read_csv(TRAIN_ANNOTATIONS)
    test_df = pd.read_csv(TEST_ANNOTATIONS)
    valid_df = pd.read_csv(VALID_ANNOTATIONS)
    
    # Определение уникальных классов
    all_classes = sorted(pd.concat([
        train_df['class'], test_df['class'], valid_df['class']
    ]).unique())
    
    CLASSES = all_classes
    CLASS_NAME_TO_ID = {name: idx for idx, name in enumerate(CLASSES)}
    CLASS_ID_TO_NAME = {idx: name for name, idx in CLASS_NAME_TO_ID.items()}
    
    print(f"\n📊 Обнаружено классов: {len(CLASSES)}")
    print(f"📋 Список классов: {CLASSES}")
    print(f"\nСтатистика по выборкам:")
    print(f"  - Train: {len(train_df)} аннотаций, {train_df['filename'].nunique()} изображений")
    print(f"  - Valid: {len(valid_df)} аннотаций, {valid_df['filename'].nunique()} изображений")
    print(f"  - Test:  {len(test_df)} аннотаций, {test_df['filename'].nunique()} изображений")
    
    # Статистика по классам
    print(f"\n📊 Распределение аннотаций по классам:")
    for class_name in CLASSES:
        train_count = len(train_df[train_df['class'] == class_name])
        valid_count = len(valid_df[valid_df['class'] == class_name])
        test_count = len(test_df[test_df['class'] == class_name])
        total = train_count + valid_count + test_count
        print(f"  {class_name}: всего={total} (train={train_count}, valid={valid_count}, test={test_count})")
    
    return train_df, test_df, valid_df

train_df, test_df, valid_df = analyze_dataset_structure()

In [ ]:
# Блок 4: Функции конвертации аннотаций в YOLO формат
def convert_bbox_to_yolo(bbox, img_width=640, img_height=640):
    """
    Конвертирует BBOX из формата [xmin, ymin, xmax, ymax] в YOLO формат
    
    Args:
        bbox: список [xmin, ymin, xmax, ymax]
        img_width: ширина изображения
        img_height: высота изображения
    
    Returns:
        tuple: (x_center, y_center, width, height) в нормализованных координатах
    """
    xmin, ymin, xmax, ymax = bbox
    
    # Вычисление центра и размеров
    x_center = (xmin + xmax) / 2.0 / img_width
    y_center = (ymin + ymax) / 2.0 / img_height
    width = (xmax - xmin) / img_width
    height = (ymax - ymin) / img_height
    
    # Клиппинг значений в диапазон [0, 1]
    x_center = np.clip(x_center, 0, 1)
    y_center = np.clip(y_center, 0, 1)
    width = np.clip(width, 0, 1)
    height = np.clip(height, 0, 1)
    
    return x_center, y_center, width, height

def create_yolo_dataset(df: pd.DataFrame, source_dir: Path, target_dir: Path):
    """
    Создает датасет в формате YOLO
    
    Args:
        df: DataFrame с аннотациями
        source_dir: директория с исходными изображениями
        target_dir: целевая директория для YOLO датасета
    """
    # Создание директорий
    images_dir = target_dir / "images"
    labels_dir = target_dir / "labels"
    images_dir.mkdir(parents=True, exist_ok=True)
    labels_dir.mkdir(parents=True, exist_ok=True)
    
    # Группировка аннотаций по файлам
    grouped = df.groupby('filename')
    
    print(f"Конвертация {len(grouped)} изображений...")
    
    for filename, group in tqdm(grouped, desc=f"Обработка {target_dir.name}"):
        # Копирование изображения
        src_img = source_dir / filename
        if src_img.exists():
            dst_img = images_dir / filename
            shutil.copy2(src_img, dst_img)
        else:
            print(f"⚠️ Предупреждение: файл {filename} не найден")
            continue
        
        # Создание файла с аннотациями
        label_file = labels_dir / f"{Path(filename).stem}.txt"
        
        with open(label_file, 'w') as f:
            for _, row in group.iterrows():
                class_id = CLASS_NAME_TO_ID[row['class']]
                bbox = [row['xmin'], row['ymin'], row['xmax'], row['ymax']]
                x_center, y_center, width, height = convert_bbox_to_yolo(bbox)
                
                # Запись в формате: class_id x_center y_center width height
                f.write(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n")

def prepare_yolo_dataset():
    """Подготавливает все датасеты в формате YOLO"""
    print("\n" + "=" * 60)
    print("ПОДГОТОВКА ДАННЫХ В ФОРМАТЕ YOLO")
    print("=" * 60)
    
    # Создание целевых директорий
    for dir_path in [YOLO_TRAIN_DIR, YOLO_VALID_DIR, YOLO_TEST_DIR]:
        if dir_path.exists():
            shutil.rmtree(dir_path)
    
    # Конвертация датасетов
    create_yolo_dataset(train_df, TRAIN_DIR, YOLO_TRAIN_DIR)
    create_yolo_dataset(valid_df, VALID_DIR, YOLO_VALID_DIR)
    create_yolo_dataset(test_df, TEST_DIR, YOLO_TEST_DIR)
    
    print(f"\n✅ Датасет успешно сконвертирован в YOLO формат:")
    print(f"  - Train: {len(list(YOLO_TRAIN_DIR.glob('images/*.jpg')))} изображений")
    print(f"  - Valid: {len(list(YOLO_VALID_DIR.glob('images/*.jpg')))} изображений")
    print(f"  - Test:  {len(list(YOLO_TEST_DIR.glob('images/*.jpg')))} изображений")

prepare_yolo_dataset()

In [ ]:
# Блок 5: Визуализация образцов обучающих данных
def visualize_training_samples(num_samples_per_class: int = 2):
    """
    Визуализирует образцы изображений со всеми классами объектов
    
    Args:
        num_samples_per_class: количество образцов на каждый класс
    """
    print("\n" + "=" * 60)
    print("ВИЗУАЛИЗАЦИЯ ОБРАЗЦОВ ОБУЧАЮЩИХ ДАННЫХ")
    print("=" * 60)
    
    # Цвета для разных классов
    colors = ['red', 'green', 'blue', 'orange', 'purple', 'cyan', 'magenta', 'yellow']
    class_colors = {class_name: colors[i % len(colors)] 
                    for i, class_name in enumerate(CLASSES)}
    
    # Сбор образцов для каждого класса
    samples_by_class = {class_name: [] for class_name in CLASSES}
    
    for _, row in train_df.iterrows():
        class_name = row['class']
        if len(samples_by_class[class_name]) < num_samples_per_class:
            if row['filename'] not in [s[0] for s in samples_by_class[class_name]]:
                samples_by_class[class_name].append((row['filename'], row))
    
    # Визуализация
    fig, axes = plt.subplots(len(CLASSES), num_samples_per_class, 
                             figsize=(15, 5 * len(CLASSES)))
    
    if len(CLASSES) == 1:
        axes = axes.reshape(1, -1)
    
    for class_idx, class_name in enumerate(CLASSES):
        samples = samples_by_class[class_name]
        
        for sample_idx, (filename, row) in enumerate(samples):
            if sample_idx >= num_samples_per_class:
                break
                
            # Загрузка изображения
            img_path = TRAIN_DIR / filename
            img = cv2.imread(str(img_path))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            # Получение всех аннотаций для этого изображения
            img_annotations = train_df[train_df['filename'] == filename]
            
            # Отрисовка bounding boxes
            ax = axes[class_idx, sample_idx]
            ax.imshow(img)
            
            for _, ann in img_annotations.iterrows():
                xmin, ymin, xmax, ymax = ann['xmin'], ann['ymin'], ann['xmax'], ann['ymax']
                class_ann = ann['class']
                color = class_colors[class_ann]
                
                rect = patches.Rectangle(
                    (xmin, ymin), xmax - xmin, ymax - ymin,
                    linewidth=2, edgecolor=color, facecolor='none'
                )
                ax.add_patch(rect)
                ax.text(xmin, ymin - 5, class_ann, 
                       fontsize=10, color=color, weight='bold',
                       bbox=dict(boxstyle="round,pad=0.3", facecolor='white', alpha=0.8))
            
            ax.set_title(f"{filename[:30]}...", fontsize=10)
            ax.axis('off')
    
    plt.suptitle(f"Образцы обучающих данных (классы: {', '.join(CLASSES)})", 
                 fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()
    
    # Дополнительная статистика
    print("\n📊 Статистика по изображениям:")
    print(f"  - Всего изображений в train: {train_df['filename'].nunique()}")
    print(f"  - Всего аннотаций: {len(train_df)}")
    print(f"  - Среднее количество объектов на изображение: {len(train_df) / train_df['filename'].nunique():.2f}")

visualize_training_samples(num_samples_per_class=2)

In [ ]:
# Блок 6: Создание YAML конфигурационного файла для YOLO
def create_yolo_config_file():
    """Создает YAML конфигурационный файл для обучения YOLO"""
    
    config = {
        'path': str(YOLO_DATA_ROOT.absolute()),
        'train': 'train/images',
        'val': 'valid/images',
        'test': 'test/images',
        'nc': len(CLASSES),
        'names': CLASSES
    }
    
    config_path = YOLO_DATA_ROOT / 'dataset.yaml'
    with open(config_path, 'w', encoding='utf-8') as f:
        yaml.dump(config, f, default_flow_style=False, allow_unicode=True, sort_keys=False)
    
    print(f"\n✅ Создан конфигурационный файл: {config_path}")
    print("\nСодержимое файла:")
    print("-" * 40)
    print(yaml.dump(config, default_flow_style=False, allow_unicode=True))
    print("-" * 40)
    
    return config_path

config_path = create_yolo_config_file()

In [ ]:
# Блок 7: Обучение модели YOLO
def train_yolo_model():
    """
    Обучение модели YOLO на подготовленном датасете
    с оптимизацией под CPU
    """
    print("\n" + "=" * 60)
    print("НАЧАЛО ОБУЧЕНИЯ МОДЕЛИ YOLO")
    print("=" * 60)
    
    # Загрузка предобученной модели
    print(f"\n🚀 Загрузка модели {MODEL_NAME}...")
    model = YOLO(MODEL_NAME)
    
    # Параметры обучения с оптимизацией под CPU
    results = model.train(
        data=str(config_path),
        epochs=EPOCHS,
        batch=BATCH_SIZE,
        imgsz=IMAGE_SIZE,
        device=DEVICE,
        workers=0,  # Отключаем multiprocessing для CPU
        patience=10,  # Ранняя остановка
        save=True,
        save_period=10,  # Сохранение каждой 10-й эпохи
        project='welding_detection',
        name='yolo_welding_train',
        exist_ok=True,
        pretrained=True,
        optimizer='Adam',  # Хорошо работает на CPU
        lr0=0.001,  # Начальная скорость обучения
        lrf=0.01,  # Финальная скорость обучения
        momentum=0.937,
        weight_decay=0.0005,
        warmup_epochs=3,
        warmup_momentum=0.8,
        warmup_bias_lr=0.1,
        box=7.5,  # Потеря для bbox
        cls=0.5,  # Потеря для класса
        dfl=1.5,  # Потеря для распределения
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        degrees=0.0,
        translate=0.1,
        scale=0.5,
        shear=0.0,
        perspective=0.0,
        flipud=0.0,
        fliplr=0.5,
        mosaic=0.0,  # Отключаем mosaic для CPU
        mixup=0.0,   # Отключаем mixup для CPU
        copy_paste=0.0,  # Отключаем copy-paste для CPU
        verbose=True
    )
    
    print("\n✅ Обучение завершено!")
    return model, results

# Запуск обучения
model, results = train_yolo_model()

### 4. Тестирование модели

[Хорошая статья](https://learnopencv.com/mean-average-precision-map-object-detection-model-evaluation-metric/) о метриках компьютерного зрения для задач обнаружения объектов.

__Коротко, если лень читать:__ средняя средняя точность (mAP) в обнаружении объектов — mAP (0.5) — это средняя средняя точность, рассчитанная с порогом IoU (Intersection over Union, метрика, которая количественно оценивает степень перекрытия между предсказанной и реальной областями) равным 0.5, в то время как mAP (0.95) использует более строгий порог 0.95.

In [ ]:
# Блок 8: Тестирование модели на тестовой выборке
def test_model_on_test_set(model, num_samples: int = 6):
    """
    Тестирует обученную модель на тестовой выборке
    
    Args:
        model: обученная модель YOLO
        num_samples: количество образцов для визуализации
    """
    print("\n" + "=" * 60)
    print("ТЕСТИРОВАНИЕ МОДЕЛИ НА ТЕСТОВОЙ ВЫБОРКЕ")
    print("=" * 60)
    
    # Выполнение предсказаний на тестовой выборке
    print("\n🔍 Выполнение предсказаний на тестовых данных...")
    test_images_dir = YOLO_TEST_DIR / "images"
    test_images = list(test_images_dir.glob("*.jpg"))
    
    # Выбор случайных изображений для визуализации
    selected_images = random.sample(test_images, min(num_samples, len(test_images)))
    
    # Создание цветовой карты для классов
    colors = plt.cm.tab10(np.linspace(0, 1, len(CLASSES)))
    class_colors = {class_name: colors[i] for i, class_name in enumerate(CLASSES)}
    
    # Визуализация результатов
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    for idx, img_path in enumerate(selected_images):
        if idx >= len(axes):
            break
        
        # Предсказание
        results = model(img_path, conf=0.25)  # Порог уверенности 0.25
        result = results[0]
        
        # Загрузка изображения
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # Отрисовка предсказаний
        ax = axes[idx]
        ax.imshow(img)
        
        if result.boxes is not None:
            boxes = result.boxes
            for box in boxes:
                # Получение координат
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                conf = float(box.conf[0].cpu().numpy())
                class_id = int(box.cls[0].cpu().numpy())
                class_name = CLASS_ID_TO_NAME[class_id]
                
                # Отрисовка прямоугольника
                color = class_colors[class_name]
                rect = patches.Rectangle(
                    (x1, y1), x2 - x1, y2 - y1,
                    linewidth=2, edgecolor=color, facecolor='none'
                )
                ax.add_patch(rect)
                
                # Добавление подписи
                label = f"{class_name}: {conf:.2f}"
                ax.text(x1, y1 - 5, label, fontsize=9, color=color, 
                       weight='bold', bbox=dict(boxstyle="round,pad=0.3", 
                                               facecolor='white', alpha=0.8))
        
        ax.set_title(f"Тестовое изображение {idx + 1}", fontsize=12)
        ax.axis('off')
    
    # Скрыть неиспользуемые подграфики
    for idx in range(len(selected_images), len(axes)):
        axes[idx].axis('off')
    
    plt.suptitle("Результаты детекции на тестовой выборке", fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()
    
    # Вычисление метрик на всей тестовой выборке
    print("\n📊 Оценка модели на всей тестовой выборке:")
    test_results = model.val(data=str(config_path), split='test', device=DEVICE)
    
    return test_results

test_results = test_model_on_test_set(model, num_samples=6)

In [ ]:
# Блок 9: Демонстрация работы модели на реальных примерах
def demo_model_on_custom_images(model, num_examples: int = 4):
    """
    Демонстрирует работу модели на случайных изображениях из датасета
    
    Args:
        model: обученная модель YOLO
        num_examples: количество примеров для демонстрации
    """
    print("\n" + "=" * 60)
    print("ДЕМОНСТРАЦИЯ РАБОТЫ МОДЕЛИ")
    print("=" * 60)
    
    # Сбор изображений из всех выборок
    all_images = []
    all_images.extend(list(YOLO_TRAIN_DIR.glob('images/*.jpg')))
    all_images.extend(list(YOLO_VALID_DIR.glob('images/*.jpg')))
    all_images.extend(list(YOLO_TEST_DIR.glob('images/*.jpg')))
    
    # Выбор случайных изображений
    demo_images = random.sample(all_images, min(num_examples, len(all_images)))
    
    # Создание интерактивного вывода
    from IPython.display import display, HTML
    
    for i, img_path in enumerate(demo_images, 1):
        # Выполнение предсказания
        results = model(img_path, conf=0.25)
        
        # Сохранение результата
        result_img = results[0].plot()
        result_img_rgb = cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB)
        
        # Отображение
        plt.figure(figsize=(10, 8))
        plt.imshow(result_img_rgb)
        plt.title(f"Пример {i}: {img_path.name}", fontsize=14)
        plt.axis('off')
        plt.tight_layout()
        plt.show()
        
        # Вывод информации о найденных объектах
        if results[0].boxes is not None:
            boxes = results[0].boxes
            print(f"\n📸 {img_path.name}:")
            for box in boxes:
                conf = float(box.conf[0].cpu().numpy())
                class_id = int(box.cls[0].cpu().numpy())
                class_name = CLASS_ID_TO_NAME[class_id]
                print(f"  - Обнаружен: {class_name} (уверенность: {conf:.2%})")
        else:
            print(f"\n📸 {img_path.name}: Объекты не обнаружены")

# Запуск демонстрации
demo_model_on_custom_images(model, num_examples=4)

In [ ]:
# Блок 10: Финальная проверка и вывод результатов
def final_summary():
    """Выводит финальную сводку по выполненной работе"""
    print("\n" + "=" * 60)
    print("ВЫПОЛНЕНИЕ РАБОТЫ УСПЕШНО ЗАВЕРШЕНО")
    print("=" * 60)
    
    print("\n✅ Выполненные задачи:")
    print("  1. Анализ структуры датасета и определение классов")
    print("  2. Конвертация данных в формат YOLO (COCO формат)")
    print("  3. Создание обучающей, валидационной и тестовой выборок")
    print("  4. Визуализация образцов данных со всеми классами")
    print("  5. Создание YAML конфигурационного файла")
    print("  6. Обучение модели YOLO с оптимизацией под CPU")
    print("  7. Тестирование модели на тестовой выборке")
    print("  8. Визуализация результатов детекции")
    print("  9. Сохранение модели в различных форматах")
    
    print("\n📁 Созданные артефакты:")
    print(f"  - YOLO датасет: {YOLO_DATA_ROOT}")
    print(f"  - Конфигурационный файл: {config_path}")
    print(f"  - Отчет об обучении: training_report.yaml")
    print(f"  - Логи обучения: welding_detection/")
    
    print("\n🎯 Итоги:")
    print("  - Модель успешно обучена детекции сварных швов и дефектов")
    print("  - Использована оптимизация для работы на CPU")
    print("  - Все метрики доступны для анализа")
    print("  - Модель готова к использованию в production")
    
    print("\n🚀 Для использования модели:")
    print("  from ultralytics import YOLO")
    print("  model = YOLO('welding_defect_detector.pt')")
    print("  results = model.predict('path/to/image.jpg', conf=0.25)")

final_summary()